# TF-IDF: поиск похожих случаев

В этом ноутбуке готовим индекс по неизменяемому корпусу и определяем функцию `find_similar_cases`. Она ищет по тексту описания, показывает top-k результатов и возвращает их в виде DataFrame без лабораторных номеров и других идентификаторов.

## 1. Импорты и определение корня проекта

Находим корень проекта, чтобы путь к корпусу не зависел от текущей папки Jupyter.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "requirements.txt").exists():
    if PROJECT_ROOT == PROJECT_ROOT.parent:
        raise FileNotFoundError(
            "Не удалось найти корень проекта с requirements.txt."
        )
    PROJECT_ROOT = PROJECT_ROOT.parent


## 2. Загрузка поискового корпуса

Загружаем только поля, нужные для поиска и показа результата. `case_id` и другие идентификаторы в DataFrame не включаем. Этот файл используется как индекс и не изменяется.

In [ ]:
DATA_PATH = PROJECT_ROOT / "data" / "processed" / "dataset_first.csv"

cases = pd.read_csv(
    DATA_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=["description", "bethesda", "conclusion"],
)


## 3. Подготовка описаний для индекса

Для поиска оставляем строки с непустым `description`. Bethesda и заключение сохраняем только как поля результата: TF-IDF строится по описанию.

In [ ]:
has_description = cases["description"].notna()
has_description &= cases["description"].str.strip().ne("")

search_cases = cases.loc[has_description].copy().reset_index(drop=True)

if search_cases.empty:
    raise ValueError("В корпусе нет непустых описаний для поиска.")


## 4. Построение TF-IDF-индекса

Используем униграммы и биграммы: отдельные слова и пары соседних слов. Векторизатор обучается на описаниях корпуса; новые запросы будут только преобразовываться в этот же векторный формат.

In [ ]:
vectorizer = TfidfVectorizer(ngram_range=(1, 2))
tfidf_matrix = vectorizer.fit_transform(search_cases["description"])


## 5. Функция поиска похожих случаев

`rank` показывает место результата в ранжированном списке, начиная с 1. `similarity` — косинусная мера сходства TF-IDF-векторов. Функция показывает таблицу в ноутбуке и одновременно возвращает её. Ручная оценка `manual_relevance` будет отдельным полем в проверочном CSV, а не частью результата поиска.

In [ ]:
def find_similar_cases(text, top_k=5):
    """Вернуть top_k случаев, похожих на описание text."""
    if not isinstance(text, str) or not text.strip():
        raise ValueError("text должен быть непустой строкой.")

    if isinstance(top_k, bool) or not isinstance(top_k, int) or top_k < 1:
        raise ValueError("top_k должен быть целым числом больше нуля.")

    query_vector = vectorizer.transform([text.strip()])
    if query_vector.nnz == 0:
        raise ValueError(
            "В запросе нет слов, известных TF-IDF-словарю корпуса."
        )

    similarities = cosine_similarity(query_vector, tfidf_matrix).ravel()
    ranked_positions = np.argsort(-similarities, kind="stable")[:top_k]

    results = search_cases.iloc[ranked_positions][
        ["description", "bethesda", "conclusion"]
    ].copy()
    results.insert(0, "similarity", similarities[ranked_positions])
    results.insert(0, "rank", np.arange(1, len(results) + 1))
    results = results.reset_index(drop=True)
    
    return results


## 6. Загрузка тестовых запросов по Bethesda

Читаем только `description` и `bethesda` из `data/raw/thyroid_bethesda_examples.csv`. Для каждой категории `I`–`VI` берём первые пять непустых описаний в порядке строк файла. Тексты не печатаются; заключение для входа в поиск не используется.

In [ ]:
TEST_EXAMPLES_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "thyroid_bethesda_examples.csv"
)
BETHESDA_CATEGORIES = ["I", "II", "III", "IV", "V", "VI"]
QUERIES_PER_CATEGORY = 5

test_examples = pd.read_csv(
    TEST_EXAMPLES_PATH,
    sep=";",
    encoding="utf-8-sig",
    usecols=["description", "bethesda"],
    dtype="string",
)

source_categories = test_examples["bethesda"].str.strip()
test_queries_by_bethesda: dict[str, list[str]] = {}

for category in BETHESDA_CATEGORIES:
    category_mask = source_categories.eq(category).fillna(False)
    descriptions = (
        test_examples.loc[category_mask, "description"]
        .dropna()
        .str.strip()
    )
    descriptions = descriptions.loc[descriptions.ne("")]
    selected_descriptions = descriptions.head(QUERIES_PER_CATEGORY).tolist()

    if len(selected_descriptions) != QUERIES_PER_CATEGORY:
        raise ValueError(
            f"Для Bethesda {category} найдено меньше "
            f"{QUERIES_PER_CATEGORY} непустых описаний."
        )

    test_queries_by_bethesda[category] = selected_descriptions


## 7. Подготовка метаданных запуска MLflow

Запуск фиксирует качественный просмотр retrieval-выдачи. Метрики качества не рассчитываем. Перед логированием укажите имя участника и Git commit.

In [ ]:
from datetime import datetime, timezone
import hashlib
import re
import tempfile
from pathlib import Path

import joblib
import mlflow
import sklearn

MLFLOW_TRACKING_URI = "http://127.0.0.1:5000"
MLFLOW_EXPERIMENT_NAME = "semantic-case-search"
PARTICIPANT_NAME = "Vildan"
GIT_COMMIT = "a79d146"

if not PARTICIPANT_NAME.strip() or PARTICIPANT_NAME.startswith("ВСТАВЬТЕ"):
    raise ValueError("Укажите имя участника перед логированием.")

if not re.fullmatch(r"[0-9a-fA-F]{7,40}", GIT_COMMIT):
    raise ValueError("Вставьте Git commit SHA (от 7 до 40 шестнадцатеричных символов).")

REVIEW_CSV_PATH = output_path
if not REVIEW_CSV_PATH.is_file():
    raise FileNotFoundError("Сначала сформируйте CSV с найденными случаями.")

EXPECTED_REVIEW_COLUMNS = [
    "query_no",
    "query_bethesda",
    "query_description",
    "rank",
    "similarity",
    "result_description",
    "result_bethesda",
    "result_conclusion",
    "manual_relevance",
    "manual_comment",
]
if list(review_table.columns) != EXPECTED_REVIEW_COLUMNS:
    raise ValueError("Проверьте состав колонок выдачи и отсутствие идентификаторов.")

def sha256_file(file_path):
    """Вычислить SHA-256 файла, не выводя его содержимое."""
    digest = hashlib.sha256()
    with Path(file_path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

RUN_DATE_UTC = datetime.now(timezone.utc).isoformat(timespec="seconds")
INDEX_SIZE_CASES = int(tfidf_matrix.shape[0])
QUERY_COUNT = sum(len(queries) for queries in test_queries_by_bethesda.values())

if INDEX_SIZE_CASES != len(search_cases):
    raise ValueError("Размер TF-IDF-индекса не совпал с размером поискового корпуса.")

vectorizer_params = vectorizer.get_params(deep=False)
parameter_names = (
    "ngram_range",
    "lowercase",
    "stop_words",
    "token_pattern",
    "norm",
    "use_idf",
    "smooth_idf",
    "sublinear_tf",
    "min_df",
    "max_df",
    "max_features",
)

RUN_PARAMETERS = {
    "participant_name": PARTICIPANT_NAME,
    "task_name": "similar_case_retrieval",
    "run_date_utc": RUN_DATE_UTC,
    "git_commit": GIT_COMMIT,
    "model_name": "TF-IDF cosine retrieval",
    "dataset_version_sha256": sha256_file(DATA_PATH),
    "query_source_sha256": sha256_file(TEST_EXAMPLES_PATH),
    "retrieval_examples_sha256": sha256_file(REVIEW_CSV_PATH),
    "index_size_cases": str(INDEX_SIZE_CASES),
    "query_count": str(QUERY_COUNT),
    "retrieved_examples_count": str(len(review_table)),
    "size_semantics": "index cases and query count; no supervised train/test split",
    "query_field": "description",
    "similarity_metric": "cosine",
    "top_k": str(TOP_K),
    "random_seed": "not_applicable; deterministic TF-IDF retrieval",
    "quality_evaluation_status": "qualitative output review; no quality metrics computed",
    "python_version": __import__("platform").python_version(),
    "scikit_learn_version": sklearn.__version__,
    "mlflow_version": mlflow.__version__,
}
RUN_PARAMETERS.update(
    {
        f"tfidf_{name}": repr(vectorizer_params[name])
        for name in parameter_names
    }
)

## 8. Логирование качественного retrieval-прогона

Сохраняем параметры и происхождение запуска, CSV с найденными случаями, а также TF-IDF-векторизатор и разреженную матрицу индекса. Метрики качества не логируются, поскольку они не рассчитывались. Артефакты отправляются только на локальный MLflow-сервер.

In [ ]:
mlflow.set_tracking_uri(MLFLOW_TRACKING_URI)
mlflow.set_experiment(MLFLOW_EXPERIMENT_NAME)

with mlflow.start_run(run_name="tfidf-qualitative-retrieval-review") as active_run:
    mlflow.set_tags(
        {
            "participant_name": PARTICIPANT_NAME,
            "task_name": "similar_case_retrieval",
            "git_commit": GIT_COMMIT,
            "quality_metrics_computed": "false",
            "evaluation_type": "qualitative_output_review",
        }
    )
    mlflow.log_params(RUN_PARAMETERS)

    with tempfile.TemporaryDirectory(prefix="tfidf_mlflow_artifacts_") as temp_dir:
        index_artifact_path = Path(temp_dir) / "tfidf_index.joblib"
        joblib.dump(
            {
                "vectorizer": vectorizer,
                "tfidf_matrix": tfidf_matrix,
                "index_size_cases": INDEX_SIZE_CASES,
                "dataset_version_sha256": RUN_PARAMETERS["dataset_version_sha256"],
            },
            index_artifact_path,
        )
        mlflow.log_artifact(str(index_artifact_path), artifact_path="model")
        mlflow.log_artifact(
            str(REVIEW_CSV_PATH),
            artifact_path="retrieved_examples",
        )

print(f"Запуск MLflow сохранён. Run ID: {active_run.info.run_id}")